# Customer Churn Prediction

# Model Training & Evaluation (v2 - Improved)

## Objective

Train and compare multiple machine learning models to predict customer churn. 
In this version, we introduce hyperparameter tuning using grid search and handle class imbalance to optimize the Recall and F1-score.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

## Load and Prepare Data

In [2]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df['TotalCharges'] = df['TotalCharges'].fillna(0)

df_ml = df.drop('customerID', axis=1)

df_ml['Churn'] = df_ml['Churn'].map({
    'No': 0,
    'Yes': 1
})

df_encoded = pd.get_dummies(
    df_ml,
    drop_first=True
)

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

num_cols = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges'
]

X_train[num_cols] = scaler.fit_transform(
    X_train[num_cols]
)

X_test[num_cols] = scaler.transform(
    X_test[num_cols]
)

## Hyperparameter Tuning & Handling Class Imbalance

In [4]:
# Calculate negative-to-positive class ratio for scale_pos_weight
ratio = sum(y_train == 0) / sum(y_train == 1)
print(f"Negative to Positive Class Ratio: {ratio:.2f}")

Negative to Positive Class Ratio: 2.77


In [5]:
# 1. Logistic Regression Tuning with class imbalance weight
lr_params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}
lr_grid = GridSearchCV(
    LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    lr_params, 
    scoring='f1',
    cv=5,
    n_jobs=-1
)
lr_grid.fit(X_train, y_train)
best_lr = lr_grid.best_estimator_
print("Best Logistic Regression Params:", lr_grid.best_params_)

lr_pred = best_lr.predict(X_test)
lr_prob = best_lr.predict_proba(X_test)[:,1]

Best Logistic Regression Params: {'C': 0.1, 'solver': 'lbfgs'}


In [6]:
# 2. Random Forest Tuning with class imbalance weight
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10]
}
rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    rf_params,
    scoring='f1',
    cv=5,
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
print("Best Random Forest Params:", rf_grid.best_params_)

rf_pred = best_rf.predict(X_test)
rf_prob = best_rf.predict_proba(X_test)[:,1]

Best Random Forest Params: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 300}


In [7]:
# 3. XGBoost Tuning with scale_pos_weight
xgb_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2]
}
xgb_grid = GridSearchCV(
    XGBClassifier(scale_pos_weight=ratio, random_state=42, eval_metric='logloss'),
    xgb_params,
    scoring='f1',
    cv=5,
    n_jobs=-1
)
xgb_grid.fit(X_train, y_train)
best_xgb = xgb_grid.best_estimator_
print("Best XGBoost Params:", xgb_grid.best_params_)

xgb_pred = best_xgb.predict(X_test)
xgb_prob = best_xgb.predict_proba(X_test)[:,1]

Best XGBoost Params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}


## Model Comparison

In [8]:
def evaluate_model(y_true, y_pred, y_prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
        'ROC AUC': roc_auc_score(y_true, y_prob)
    }

In [9]:
results = pd.DataFrame({
    'Logistic Regression (Tuned)': evaluate_model(y_test, lr_pred, lr_prob),
    'Random Forest (Tuned)': evaluate_model(y_test, rf_pred, rf_prob),
    'XGBoost (Tuned)': evaluate_model(y_test, xgb_pred, xgb_prob)
}).T

print(results.sort_values(by='F1 Score', ascending=False))

                             Accuracy  Precision    Recall  F1 Score   ROC AUC
Random Forest (Tuned)        0.760823   0.533698  0.783422  0.634886  0.842227
XGBoost (Tuned)              0.747339   0.515517  0.799465  0.626834  0.840439
Logistic Regression (Tuned)  0.743080   0.510453  0.783422  0.618143  0.841288


## Save Best Model

In [10]:
best_model_name = results.sort_values(by='F1 Score', ascending=False).index[0]
print(f"Selected Best Model: {best_model_name}")

if "Logistic Regression" in best_model_name:
    best_model = best_lr
elif "Random Forest" in best_model_name:
    best_model = best_rf
else:
    best_model = best_xgb

os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/churn_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print(f"Best model '{best_model_name}' Saved Successfully to '../models/churn_model.pkl'")

Selected Best Model: Random Forest (Tuned)


Best model 'Random Forest (Tuned)' Saved Successfully to '../models/churn_model.pkl'

## Best Model Confusion Matrix

In [11]:
best_model_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_model_pred)

plt.figure(figsize=(6,4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)
plt.title(f'{best_model_name} Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

C:\Users\Mohamed\AppData\Local\Temp\ipykernel_28560\890606561.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Classification Report

In [12]:
print(classification_report(y_test, best_model_pred))

              precision    recall  f1-score   support

           0       0.91      0.75      0.82      1035
           1       0.53      0.78      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.77      0.73      1409
weighted avg       0.81      0.76      0.77      1409



## ROC Curve

In [13]:
best_model_prob = best_model.predict_proba(X_test)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, best_model_prob)
auc_score = roc_auc_score(y_test, best_model_prob)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f'{best_model_name} (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'{best_model_name} ROC Curve')
plt.legend(loc='lower right')
plt.show()

C:\Users\Mohamed\AppData\Local\Temp\ipykernel_28560\4047676148.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
